1. Đọc một mẫu có đáp án và một mẫu không có đáp án

In [1]:
import json
from pathlib import Path

# Nếu notebook đang ở thư mục notebooks/, lùi lên thư mục gốc dự án.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

train_path = project_root / "data" / "processed" / "train.json"

answerable_example = None
unanswerable_example = None

# Tìm mỗi loại một mẫu trong tập train con.
with train_path.open("r", encoding="utf-8") as input_file:
    for line in input_file:
        example = json.loads(line)
        has_answer = len(example["answers"]["text"]) > 0

        if not example["is_impossible"] and has_answer:
            answerable_example = example

        if example["is_impossible"] or not has_answer:
            unanswerable_example = example

        if answerable_example is not None and unanswerable_example is not None:
            break

print("Mẫu có đáp án:", answerable_example["answers"]["text"][0])
print("Mẫu không có đáp án:", unanswerable_example["is_impossible"])

Mẫu có đáp án: hoạt động văn chương, âm nhạc, nghệ thuật và sân khấu
Mẫu không có đáp án: True


2. Tải fast tokenizer và import các hàm

In [2]:
import sys
from pathlib import Path
from transformers import AutoTokenizer

# Đường dẫn gốc dự án bạn đã cung cấp.
project_root = Path(
    r"C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project"
)

# Kiểm tra Python có nhìn thấy file cần import không.
preprocessing_file = project_root / "src" / "data_preprocessing.py"
print("Đang tìm file:", preprocessing_file)
assert preprocessing_file.exists(), "Không tìm thấy data_preprocessing.py ở đường dẫn này."

# Thêm thư mục gốc dự án vào danh sách nơi Python tìm module.
sys.path.insert(0, str(project_root))

# Bây giờ Python có thể tìm package src.
from src.data_preprocessing import (
    prepare_train_features,
    prepare_validation_features,
)

#khởi tạo tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "xlm-roberta-base",
    use_fast=True,
)


print("Import thành công")
print("Tokenizer đã tải:", tokenizer.name_or_path)
print("Fast tokenizer:", tokenizer.is_fast)

c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang tìm file: C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\src\data_preprocessing.py
Import thành công
Tokenizer đã tải: xlm-roberta-base
Fast tokenizer: True


3. Kiểm tra ánh xạ đáp án từ ký tự sang token

In [3]:
# Hàm nhận một mẫu riêng lẻ nhưng đóng gói các trường thành batch một phần tử.
def make_batch(example):
    return {
        "id": [example["id"]],
        "question": [example["question"].lstrip()],
        "context": [example["context"]],
        "answers": [example["answers"]],
        "is_impossible": [example["is_impossible"]],
    }

answerable_batch = make_batch(answerable_example)

# Gọi hàm tiền xử lý để lấy nhãn token bắt đầu và kết thúc.
train_features = prepare_train_features(answerable_batch, tokenizer)

# Tokenize lại cùng mẫu để xem offset ký tự đối chiếu với nhãn.
raw_features = tokenizer(
    answerable_batch["question"],
    answerable_batch["context"],
    truncation="only_second",
    max_length=384,
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding="max_length",
)

answer = answerable_example["answers"]["text"][0]
answer_start = answerable_example["answers"]["answer_start"][0]
answer_end = answer_start + len(answer)

# Xác nhận offset ký tự trong dữ liệu trích đúng chuỗi đáp án.
assert answerable_example["context"][answer_start:answer_end] == answer

# Tìm cửa sổ có nhãn đáp án thay vì nhãn CLS.
answer_span_found = False

for feature_index, (start_token, end_token) in enumerate(
    zip(train_features["start_positions"], train_features["end_positions"])
):
    input_ids = raw_features["input_ids"][feature_index]
    cls_index = input_ids.index(tokenizer.cls_token_id)

    # Nhãn CLS nghĩa là cửa sổ này không chứa trọn đáp án.
    if start_token == cls_index and end_token == cls_index:
        continue

    offsets = raw_features["offset_mapping"][feature_index]
    sequence_ids = raw_features.sequence_ids(feature_index)

    # Hai đầu của đáp án phải thuộc context.
    assert sequence_ids[start_token] == 1
    assert sequence_ids[end_token] == 1

    # Offset token phải bao phủ khoảng ký tự của đáp án.
    assert offsets[start_token][0] <= answer_start
    assert offsets[end_token][1] >= answer_end

    answer_span_found = True
    print("Đáp án chuẩn:", answer)
    print("Chỉ số token bắt đầu:", start_token)
    print("Chỉ số token kết thúc:", end_token)
    print("Offset token bắt đầu:", offsets[start_token])
    print("Offset token kết thúc:", offsets[end_token])
    break

# Một mẫu có đáp án phải được tìm thấy trong ít nhất một cửa sổ.
assert answer_span_found
print("Kiểm tra ánh xạ đáp án: đạt")

Đáp án chuẩn: hoạt động văn chương, âm nhạc, nghệ thuật và sân khấu
Chỉ số token bắt đầu: 55
Chỉ số token kết thúc: 67
Offset token bắt đầu: (179, 183)
Offset token kết thúc: (228, 232)
Kiểm tra ánh xạ đáp án: đạt


4. Kiểm tra câu không có đáp án và validation

In [4]:
# Với câu không có đáp án, mọi cửa sổ phải mang nhãn CLS ở cả hai đầu.
unanswerable_batch = make_batch(unanswerable_example)
no_answer_features = prepare_train_features(unanswerable_batch, tokenizer)

for feature_index, input_ids in enumerate(no_answer_features["input_ids"]):
    cls_index = input_ids.index(tokenizer.cls_token_id)

    assert no_answer_features["start_positions"][feature_index] == cls_index
    assert no_answer_features["end_positions"][feature_index] == cls_index

print("Kiểm tra nhãn câu không có đáp án: đạt")

# Validation phải giữ ID mẫu và offset của context.
validation_features = prepare_validation_features(answerable_batch, tokenizer)

assert all(
    example_id == answerable_example["id"]
    for example_id in validation_features["example_id"]
)

for offsets in validation_features["offset_mapping"]:
    assert any(offset is not None for offset in offsets)

print("Kiểm tra metadata validation: đạt")

Kiểm tra nhãn câu không có đáp án: đạt
Kiểm tra metadata validation: đạt
